# 07 指标计算模块 (core.metrics) 完整示例

覆盖分类/特征评估/稳定性/金融风控/回归/分箱统计所有指标。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
import hscredit as hsc
from hscredit import germancredit, init_setting
init_setting()
df = germancredit()
target = 'creditability'
num_feats = df.select_dtypes('number').columns.drop(target).tolist()
y = df[target].values
X = df[num_feats]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
from hscredit import WOEEncoder, LogisticRegression
woe = WOEEncoder(max_n_bins=5); X_woe = woe.fit_transform(df[num_feats], df[target])
lr = LogisticRegression(max_iter=1000); lr.fit(X_woe, df[target])
y_prob = lr.predict_proba(X_woe)[:,1]

## 1. 分类指标 (KS / AUC / Gini / Accuracy / F1)

In [ ]:
from hscredit.core.metrics import (
    ks, auc, gini, accuracy, precision, recall, f1,
    ks_bucket, roc_curve, confusion_matrix, classification_report
)
print(f'KS:        {ks(y, y_prob):.4f}')
print(f'AUC:       {auc(y, y_prob):.4f}')
print(f'Gini:      {gini(y, y_prob):.4f}')
y_pred = (y_prob > 0.3).astype(int)
print(f'Accuracy:  {accuracy(y, y_pred):.4f}')
print(f'Precision: {precision(y, y_pred):.4f}')
print(f'Recall:    {recall(y, y_pred):.4f}')
print(f'F1:        {f1(y, y_pred):.4f}')
print('\nKS分箱表:')
ks_bucket(y, y_prob, n=10).head()

## 2. 特征评估 (IV / chi2 / cramers_v)

In [ ]:
from hscredit.core.metrics import iv, iv_table, chi2_test, cramers_v, IV, IV_table
feat = df['duration.in.month'].values
print(f'IV:        {iv(y, feat):.4f}')
print(f'chi2_test: {chi2_test(feat, y)}')
print(f'cramers_v: {cramers_v(feat, y):.4f}')
iv_table(y, feat, method='mdlp', max_n_bins=6)

## 3. 稳定性 (PSI / CSI / batch_psi)

In [ ]:
from hscredit.core.metrics import psi, psi_table, psi_rating, csi, csi_table, batch_psi
np.random.seed(42)
base = df['credit.amount'].values
shift = base + np.random.normal(0, 500, len(base))  # 模拟漂移
print(f'PSI: {psi(base, shift):.4f}')
print(f'评级: {psi_rating(psi(base, shift))}')
psi_table(base, shift, max_n_bins=8)

In [ ]:
# 批量PSI
batch_result = batch_psi(df[num_feats[:3]], df[num_feats[:3]] + np.random.normal(0, 0.1, (len(df),3)))
batch_result

## 4. 金融风控指标 (Lift / BadRate / Vintage)

In [ ]:
from hscredit.core.metrics.finance import lift_at, lift_monotonicity_check, rule_lift_table
print(f'Lift@10%: {lift_at(y, y_prob, ratios=0.1):.4f}')
print(f'Lift@20%: {lift_at(y, y_prob, ratios=0.2):.4f}')
print(f'单调性检查: {lift_monotonicity_check(y, y_prob)}')
rule_hit = y_prob > 0.4
rule_lift_table(y, rule_hit)

## 5. 回归指标

In [ ]:
from hscredit.core.metrics import mse, mae, rmse, r2, MSE, MAE, RMSE, R2
np.random.seed(42)
y_r = np.random.randn(500); yhat_r = y_r + np.random.randn(500)*0.5
print(f'MSE:  {mse(y_r, yhat_r):.4f}')
print(f'MAE:  {mae(y_r, yhat_r):.4f}')
print(f'RMSE: {rmse(y_r, yhat_r):.4f}')
print(f'R2:   {r2(y_r, yhat_r):.4f}')

## 6. KS / AUC / PSI 大写别名（向后兼容）

In [ ]:
from hscredit import KS, AUC, Gini, PSI, IV, PSI_table, CSI_table, IV_table
print('KS:', KS(y, y_prob))
print('AUC:', AUC(y, y_prob))
print('Gini:', Gini(y, y_prob))
print('PSI:', PSI(base, shift))
print('IV:', IV(y, df['duration.in.month'].values))